In [ ]:
# IL-only GPU AdamW (PTX, khong C++) - 1/4: GPU + mono (KHONG build C/C++)
import os, subprocess, glob, re
!nvidia-smi
r=subprocess.run('apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mono-complete', shell=True, capture_output=True, text=True, timeout=1200)
print('mono install rc=', r.returncode, (r.stderr or '')[-400:])
v=subprocess.run('mono --version', shell=True, capture_output=True, text=True)
print((v.stdout or '').split('\n')[0])
print('KHONG import torch - duong IL-only doc lap')

In [ ]:
# 2/3: clone backend + symlink cudart/cublas/cuda(driver)
import os, subprocess, glob
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD',shell=True).decode().strip())
libdir='/content/tkvlib'; os.makedirs(libdir, exist_ok=True)
def pick(pats):
    for pat in pats:
        found=[p for p in glob.glob(pat) if os.path.exists(os.path.realpath(p))]
        if found: return sorted(found)[0]
    return None
cudart=pick(['/usr/local/cuda*/lib64/libcudart.so','/usr/lib/x86_64-linux-gnu/libcudart.so*'])
cublas=pick(['/usr/local/cuda*/lib64/libcublas.so','/usr/lib/x86_64-linux-gnu/libcublas.so*'])
cudrv=pick(['/usr/lib64-nvidia/libcuda.so*','/usr/lib/x86_64-linux-gnu/libcuda.so*','/usr/local/cuda/compat/libcuda.so*'])
print('cudart=',cudart); print('cublas=',cublas); print('cuda driver=',cudrv)
for nm,tgt in [('cudart.dll',cudart),('cublas.dll',cublas),('cuda.dll',cudrv)]:
    if tgt is None: continue
    p=libdir+'/'+nm
    if os.path.lexists(p): os.remove(p)
    os.symlink(tgt,p); print('symlink',nm,'->',tgt)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
print('LD_LIBRARY_PATH=',os.environ['LD_LIBRARY_PATH'][:160])

In [ ]:
# 3/4: chay attention backward IL-only
import subprocess, os, re, math
exe=d+'/Release/cuda_backend/examples/ilgpu_dattn.exe'
print('exe exists=',os.path.exists(exe),' bytes=',os.path.getsize(exe) if os.path.exists(exe) else 0)
r=subprocess.run(['mono',exe], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('mono rc=',r.returncode)
print(r.stdout)
print('STDERR:',(r.stderr or '')[:1000])

In [ ]:
# 4/4: reference Python thuan
def vals(tag): return [float(v) for v in re.findall(r'^'+tag+r'\s+\d+\s+([0-9.eE+-]+)$',r.stdout,re.M)]
T,D=4,8
Q=[[1.0+0.1*(i+k) for k in range(D)] for i in range(T)]
K=[[0.5+0.05*(i-k) for k in range(D)] for i in range(T)]
V=[[0.2+0.03*(i+k) for k in range(D)] for i in range(T)]
dO=[[0.1+0.02*k for k in range(D)] for i in range(T)]
S=[[0.35355339*sum(Q[i][k]*K[j][k] for k in range(D)) for j in range(T)] for i in range(T)]
E=[[math.exp(S[i][j]) if j<=i else 0.0 for j in range(T)] for i in range(T)]
P=[[E[i][j]/sum(E[i]) for j in range(T)] for i in range(T)]
dP=[[sum(dO[i][k]*V[j][k] for k in range(D)) for j in range(T)] for i in range(T)]
z=[sum(P[i][j]*dP[i][j] for j in range(T)) for i in range(T)]
dS=[[P[i][j]*(dP[i][j]-z[i]) for j in range(T)] for i in range(T)]
dQ=[[0.35355339*sum(dS[i][j]*K[j][k] for j in range(T)) for k in range(D)] for i in range(T)]
dK=[[0.35355339*sum(dS[j][i]*Q[j][k] for j in range(T)) for k in range(D)] for i in range(T)]
dV=[[sum(P[j][i]*dO[j][k] for j in range(T)) for k in range(D)] for i in range(T)]
def flat(X): return [X[r][c] for c in range(len(X[0])) for r in range(len(X))]
errs=[]
for tag,ref in [('DQ',dQ),('DK',dK),('DV',dV)]:
    got=vals(tag); rr=flat(ref)
    errs.append(max(abs(a-b) for a,b in zip(got,rr)) if len(got)==len(rr) else None)
print('DQ/DK/DV maxerr=',errs)
print('KET LUAN:', 'OK' if all(e is not None and e<2e-4 for e in errs) else 'XEM OUTPUT')
print('--- P/DS comparison ---')
gp=vals('P'); gs=vals('DS')
rp=[P[r][c] for c in range(4) for r in range(4)]
rs=[dS[r][c] for c in range(4) for r in range(4)]
print('P err=', max(abs(a-b) for a,b in zip(gp,rp)) if len(gp)==len(rp) else None, len(gp),len(rp))
print('DS err=', max(abs(a-b) for a,b in zip(gs,rs)) if len(gs)==len(rs) else None, len(gs),len(rs))